# xnor-guard Egitim (T4)
Notebook erisimi acik 3 secret: KAGGLE_USERNAME=suwasted, KAGGLE_KEY, RF_KEY
Hicbir key asagida YOKTUR.

In [ ]:
!pip install -q ultralytics roboflow kaggle
from google.colab import userdata
import os
os.environ['KAGGLE_USERNAME'] = userdata.get('KAGGLE_USERNAME')
os.environ['KAGGLE_KEY'] = userdata.get('KAGGLE_KEY')
os.environ['RF_KEY'] = userdata.get('RF_KEY')
print('secrets OK')

In [ ]:
from pathlib import Path
Path('/content/datasets').mkdir(parents=True, exist_ok=True)
print(Path('/content').exists())

## A. Kaggle: weapons-detection-raw (~2GB, 1-2 dk)

In [ ]:
!kaggle datasets download -d apex25/weapons-detection-raw -p /content/datasets
import zipfile
zipfile.ZipFile('/content/datasets/weapons-detection-raw.zip').extractall('/content/datasets/weapons-detection-raw')
import os
for r, d, f in os.walk('/content/datasets/weapons-detection-raw'):
    print(r)

In [ ]:
!mkdir -p /content/Gun-Detect-1
from roboflow import Roboflow
rf = Roboflow(api_key=os.environ['RF_KEY'])
rf.workspace('new-workspace-pnwnb').project('gun-detect-y1ogv').version(1).download('yolov8', location='/content/Gun-Detect-1', overwrite=True)
rf.workspace('joao-assalim-xmovq').project('weapon-2').version(2).download('yolov8', location='/content/datasets/weapon-2', overwrite=True)
print('roboflow weapon OK')

In [ ]:
rf.workspace('bahria-university-g0y7w').project('crime-dp3x3').version(10).download('yolov8', location='/content/datasets/crime-10', overwrite=True)
print('roboflow crime OK')

In [ ]:
rf.workspace('smoking-pwchg').project('cigarette-smokers').version(1).download('yolov8', location='/content/datasets/cigarette-smokers', overwrite=True)
rf.workspace('objectdetection-rrcdv').project('smoking-detection-dataset').version(4).download('yolov8', location='/content/datasets/smoking-detection-dataset', overwrite=True)
print('roboflow smoke OK')

## B. Scriptler (/content'e yazilir)

In [ ]:
from pathlib import Path
files = {
    'prepare_dataset.py': '"""YOLO formatindaki birden cok veri kaynagini tek sinifa harmanlar, kalite filtreler ve train/val/test ayirir.\n\nKullanim:\n    python training/prepare_dataset.py --config training/weapon_sources.json --out training/data/weapon_v2 --class-name weapon\n\nCikti:\n    <out>/images/{train,val,test}\n    <out>/labels/{train,val,test}\n    <out>/data.yaml\n    <out>/report.json\n"""\nimport argparse\nimport json\nimport random\nimport shutil\nimport sys\nfrom pathlib import Path\n\nIMG_EXTS = {".jpg", ".jpeg", ".png", ".bmp"}\n\n\ndef load_config(path: str) -> dict:\n    with open(path, encoding="utf-8") as f:\n        return json.load(f)\n\n\ndef resolve(root: Path, p: str) -> Path:\n    pp = Path(p)\n    return pp if pp.is_absolute() else (root / pp)\n\n\ndef parse_label(text: str, class_map: dict, min_box: float):\n    """Etiket satirini cozumler. (cls, cx, cy, w, h) veya None doner."""\n    parts = text.split()\n    if len(parts) != 5:\n        return None\n    try:\n        cls = int(float(parts[0]))\n        vals = [float(x) for x in parts[1:]]\n    except ValueError:\n        return None\n    if cls not in class_map:\n        return None\n    cx, cy, w, h = vals\n    if not (0.0 <= cx <= 1.0 and 0.0 <= cy <= 1.0 and 0.0 <= w <= 1.0 and 0.0 <= h <= 1.0):\n        return None\n    if w < min_box or h < min_box:\n        return None\n    return (class_map[cls], cx, cy, w, h)\n\n\ndef find_images(img_dir: Path):\n    return sorted([p for p in img_dir.rglob("*") if p.suffix.lower() in IMG_EXTS])\n\n\ndef main():\n    ap = argparse.ArgumentParser()\n    ap.add_argument("--config", required=True)\n    ap.add_argument("--out", required=True)\n    ap.add_argument("--class-name", default="weapon")\n    ap.add_argument("--min-box", type=float, default=0.002)\n    ap.add_argument("--keep-empty", action="store_true")\n    args = ap.parse_args()\n\n    cfg = load_config(args.config)\n    out = Path(args.out)\n    report = {"class": args.class_name, "sources": [], "totals": {"train": 0, "val": 0, "test": 0}}\n\n    if out.exists():\n        shutil.rmtree(out)\n    out.mkdir(parents=True)\n\n    for split in ("train", "val", "test"):\n        for sub in ("images", "labels"):\n            (out / sub / split).mkdir(parents=True, exist_ok=True)\n\n    for src in cfg["sources"]:\n        name = src["name"]\n        root = Path(src["path"])\n        if not root.is_dir():\n            print(f"[SKIP] {name}: \'{src[\'path\']}\' yok")\n            continue\n        class_map = {int(k): int(v) for k, v in src["class_map"].items()}\n        src_report = {"name": name, "images": 0, "boxes": 0, "classes": {}, "dropped_empty": 0, "dropped_bad": 0, "splits": {}}\n\n        splits_from_dirs = {s.get("split", s["dir"].lower()): s["dir"] for s in src.get("dirs", [])}\n\n        items = []\n        for raw_split, d in splits_from_dirs.items():\n            img_dir = resolve(root, d) / "images"\n            lab_dir = resolve(root, d) / "labels"\n            if not img_dir.is_dir():\n                print(f"[WARN] {name}: \'{img_dir}\' yok")\n                continue\n            for img_p in find_images(img_dir):\n                lbl_p = lab_dir / (img_p.stem + ".txt")\n                if not lbl_p.is_file():\n                    continue\n                items.append((raw_split, img_p, lbl_p))\n\n        resplit = src.get("resplit")\n        if resplit:\n            random.Random(resplit.get("seed", 42)).shuffle(items)\n            fracs = resplit.get("frac", {})\n            n = len(items)\n            n_train = int(n * fracs.get("train", 0.8))\n            n_val = int(n * fracs.get("val", 0.1))\n            assignments = ["train"] * n_train + ["val"] * n_val + ["test"] * (n - n_train - n_val)\n            items = [(assignments[i], img_p, lbl_p) for i, (_, img_p, lbl_p) in enumerate(items)]\n\n        for raw_split, img_p, lbl_p in items:\n            out_split = src.get("override", {}).get(raw_split, raw_split)\n            out_split = "val" if out_split == "valid" else out_split\n            if out_split not in ("train", "val", "test"):\n                continue\n            with open(lbl_p, encoding="utf-8", errors="replace") as f:\n                text = f.read()\n            boxes = []\n            if text.strip():\n                for line in text.splitlines():\n                    b = parse_label(line.strip(), class_map, args.min_box)\n                    if b is None:\n                        src_report["dropped_bad"] += 1\n                    else:\n                        boxes.append(b)\n            if not boxes and not args.keep_empty:\n                src_report["dropped_empty"] += 1\n                continue\n            dst_stem = f"{name}__{img_p.stem}"\n            dst_img = out / "images" / out_split / (dst_stem + img_p.suffix.lower())\n            shutil.copy2(img_p, dst_img)\n            with open(out / "labels" / out_split / f"{dst_stem}.txt", "w", encoding="utf-8") as f:\n                for b in boxes:\n                    f.write(f"{b[0]} {b[1]:.6f} {b[2]:.6f} {b[3]:.6f} {b[4]:.6f}\\n")\n            src_report["images"] += 1\n            src_report["boxes"] += len(boxes)\n            src_report["splits"][out_split] = src_report["splits"].get(out_split, 0) + 1\n            for b in boxes:\n                src_report["classes"][b[0]] = src_report["classes"].get(b[0], 0) + 1\n\n        report["sources"].append(src_report)\n        for s, v in src_report["splits"].items():\n            report["totals"][s] += v\n\n    yaml_path = out / "data.yaml"\n    yaml_path.write_text(\n        f"path: {out.resolve().as_posix()}\\n"\n        "train: images/train\\nval: images/val\\n"\n        "names:\\n  0: " + args.class_name + "\\n",\n        encoding="utf-8",\n    )\n    (out / "report.json").write_text(json.dumps(report, ensure_ascii=False, indent=2), encoding="utf-8")\n    print(json.dumps(report, ensure_ascii=False, indent=2))\n\n\nif __name__ == "__main__":\n    sys.exit(main())',
    'train_yolo.py': '"""YOLOv8/11 egitim CLI - Colab T4 icin hazir.\n\nKullanim (Colab\'de):\n    !python training/train_yolo.py --data training/data/weapon_v2/data.yaml --model yolov8n --epochs 80 --imgsz 640 --batch auto --device 0 --name weapon_v2\n\nGPUsuz calismada --device cpu. Model ciktisi: runs/<name>/weights/best.pt\n"""\nimport argparse\nimport gc\nimport json\nimport sys\n\nfrom ultralytics import YOLO\n\n\ndef main():\n    ap = argparse.ArgumentParser()\n    ap.add_argument("--data", required=True)\n    ap.add_argument("--model", default="yolov8n")\n    ap.add_argument("--epochs", type=int, default=80)\n    ap.add_argument("--imgsz", type=int, default=640)\n    ap.add_argument("--batch", default=-1)\n    ap.add_argument("--device", default=0)\n    ap.add_argument("--name", default="run")\n    ap.add_argument("--project", default="runs")\n    ap.add_argument("--seed", type=int, default=42)\n    ap.add_argument("--fraction", type=float, default=1.0,\n                    help="Veri setinin egitim icin kullanilacak orani (0-1). Hackathon hizi icin 0.3-0.5.")\n    args = ap.parse_args()\n\n    raw_batch = str(args.batch).strip().lower()\n    batch = -1 if raw_batch in ("auto", "-1") else int(float(raw_batch))\n\n    m = YOLO(args.model)\n    results = m.train(\n        data=args.data,\n        epochs=args.epochs,\n        imgsz=args.imgsz,\n        batch=batch,\n        device=args.device,\n        project=args.project,\n        name=args.name,\n        seed=args.seed,\n        fraction=args.fraction,\n        plots=True,\n    )\n\n    best_path = f"{args.project}/{args.name}/weights/best.pt"\n    summary = {\n        "best": best_path,\n        "metrics": {k: float(v) for k, v in results.results_dict.items()},\n    }\n    with open(f"{args.project}/{args.name}/training_summary.json", "w", encoding="utf-8") as f:\n        json.dump(summary, f, ensure_ascii=False, indent=2)\n    print(json.dumps(summary, ensure_ascii=False, indent=2))\n    gc.collect()\n\n\nif __name__ == "__main__":\n    sys.exit(main())',
    'weapon_sources.json': '{\n  "_docs": "Her kaynak: path (root), dirs (her satir: split adi + \'dir\' = img/labels iceren grup klasoru), class_map (kaynak sinif id -> hedef 0=weapon). weapons-raw sinif id\'leri dogrulandi (pistol=0 rifle=1 knife=2 grenade=3 heavy_gun=4 person=5; person(5) dusuyor). weapon-2 Roboflow YOLOv8 indirmesi (guns=0 knife=1, ikisi de -> weapon). gun_detect_1 sabanci klasor.",\n  "sources": [\n    {\n      "name": "gun_detect_1",\n      "path": "Gun-Detect-1",\n      "dirs": [\n        {\n          "split": "train",\n          "dir": "train"\n        },\n        {\n          "split": "valid",\n          "dir": "valid"\n        }\n      ],\n      "class_map": {\n        "0": 0,\n        "1": 0,\n        "2": 0\n      }\n    },\n    {\n      "name": "weapon2_rf",\n      "path": "../datasets/weapon-2",\n      "dirs": [\n        {\n          "split": "train",\n          "dir": "train"\n        },\n        {\n          "split": "test",\n          "dir": "test"\n        }\n      ],\n      "class_map": {\n        "0": 0,\n        "1": 0\n      },\n      "resplit": {\n        "seed": 42,\n        "frac": {\n          "train": 0.8,\n          "val": 0.1,\n          "test": 0.1\n        }\n      }\n    },\n    {\n      "name": "weapons_raw",\n      "path": "../datasets/weapons-detection-raw/merged",\n      "dirs": [\n        {\n          "split": "train",\n          "dir": "train"\n        },\n        {\n          "split": "val",\n          "dir": "val"\n        },\n        {\n          "split": "test",\n          "dir": "test"\n        }\n      ],\n      "class_map": {\n        "0": 0,\n        "1": 0,\n        "2": 0,\n        "3": 0,\n        "4": 0\n      }\n    }\n  ]\n}',
    'weapon3_sources.json': '{\n  "_docs": "weapon_v3 icin: weapon_sources.json\'daki 3 kaynak + Roboflow crime (bahria-university-g0y7w/crime-dp3x3 v10). crime sinif id\'leri data.yaml\'dan dogrulandi: 0=criminal 1=person 2=weapon -> sadece weapon(2) hedef 0\'a; criminal/person dusuyor. Yollar go relidir; Colab notebook bunlari mutlak /content/... yapar.",\n  "sources": [\n    {\n      "name": "gun_detect_1",\n      "path": "Gun-Detect-1",\n      "dirs": [\n        {\n          "split": "train",\n          "dir": "train"\n        },\n        {\n          "split": "valid",\n          "dir": "valid"\n        }\n      ],\n      "class_map": {\n        "0": 0,\n        "1": 0,\n        "2": 0\n      }\n    },\n    {\n      "name": "weapon2_rf",\n      "path": "../datasets/weapon-2",\n      "dirs": [\n        {\n          "split": "train",\n          "dir": "train"\n        },\n        {\n          "split": "test",\n          "dir": "test"\n        }\n      ],\n      "class_map": {\n        "0": 0,\n        "1": 0\n      },\n      "resplit": {\n        "seed": 42,\n        "frac": {\n          "train": 0.8,\n          "val": 0.1,\n          "test": 0.1\n        }\n      }\n    },\n    {\n      "name": "weapons_raw",\n      "path": "../datasets/weapons-detection-raw/merged",\n      "dirs": [\n        {\n          "split": "train",\n          "dir": "train"\n        },\n        {\n          "split": "val",\n          "dir": "val"\n        },\n        {\n          "split": "test",\n          "dir": "test"\n        }\n      ],\n      "class_map": {\n        "0": 0,\n        "1": 0,\n        "2": 0,\n        "3": 0,\n        "4": 0\n      }\n    },\n    {\n      "name": "crime",\n      "path": "../datasets/crime-10",\n      "dirs": [\n        {\n          "split": "train",\n          "dir": "train"\n        },\n        {\n          "split": "valid",\n          "dir": "valid"\n        },\n        {\n          "split": "test",\n          "dir": "test"\n        }\n      ],\n      "class_map": {\n        "2": 0\n      }\n    }\n  ]\n}',
    'smoking_sources.json': '{\n  "_docs": "Sigara icin tek hedef sinif: cigarette (0). Her kaynak indirildikten sonra README.roboflow.txt / data.yaml icindeki sinif sirayina gore class_map dogrulanmali; sira degisirse class_map guncelle.",\n  "sources": [\n    {\n      "name": "rf_smoking_detection",\n      "path": "../datasets/smoking-detection-dataset",\n      "dirs": [\n        {\n          "split": "train",\n          "dir": "train"\n        },\n        {\n          "split": "valid",\n          "dir": "valid"\n        },\n        {\n          "split": "test",\n          "dir": "test"\n        }\n      ],\n      "class_map": {\n        "1": 0\n      }\n    },\n    {\n      "name": "rf_cigarette_smokers",\n      "path": "../datasets/cigarette-smokers",\n      "dirs": [\n        {\n          "split": "train",\n          "dir": "train"\n        },\n        {\n          "split": "valid",\n          "dir": "valid"\n        },\n        {\n          "split": "test",\n          "dir": "test"\n        }\n      ],\n      "class_map": {\n        "0": 0\n      }\n    }\n  ]\n}',
}
base = Path('/content')
for name, txt in files.items():
    (base / name).write_text(txt, encoding='utf-8')
for name in ('weapon_sources.json', 'weapon3_sources.json', 'smoking_sources.json'):
    import json as _json
    _p = base / name
    _c = _json.loads(_p.read_text(encoding='utf-8'))
    for _s in _c['sources']:
        if _s['path'].startswith('../datasets/'):
            _s['path'] = '/content/datasets/' + _s['path'].split('datasets/', 1)[1]
        elif _s['path'] == 'Gun-Detect-1':
            _s['path'] = '/content/Gun-Detect-1'
    _p.write_text(_json.dumps(_c, ensure_ascii=False, indent=2), encoding='utf-8')
print('scripts yazildi (mutlak yollar)')
!python -m py_compile /content/prepare_dataset.py /content/train_yolo.py && echo py_compile OK


## C. Harman (weapon_v2 + weapon_v3 + cigarette_v2)

In [ ]:
!python /content/prepare_dataset.py --config /content/weapon_sources.json --out /content/data/weapon_v2 --class-name weapon
import json
print(json.dumps(json.load(open('/content/data/weapon_v2/report.json')), ensure_ascii=False, indent=2))

In [ ]:
!python /content/prepare_dataset.py --config /content/weapon3_sources.json --out /content/data/weapon_v3 --class-name weapon
import json
print(json.dumps(json.load(open('/content/data/weapon_v3/report.json')), ensure_ascii=False, indent=2))

In [ ]:
!python /content/prepare_dataset.py --config /content/smoking_sources.json --out /content/data/cigarette_v2 --class-name cigarette
import json
print(json.dumps(json.load(open('/content/data/cigarette_v2/report.json')), ensure_ascii=False, indent=2))

## D. Egitim (T4, ~1.5-3 saat/set)

In [ ]:
!python /content/train_yolo.py --data /content/data/weapon_v2/data.yaml --model yolov8s --epochs 80 --imgsz 640 --batch auto --device 0 --name weapon_v2

In [ ]:
!python /content/train_yolo.py --data /content/data/weapon_v3/data.yaml --model yolov8n --epochs 40 --imgsz 640 --batch auto --fraction 0.4 --device 0 --name weapon_v3

In [ ]:
!python /content/train_yolo.py --data /content/data/cigarette_v2/data.yaml --model yolov8s --epochs 60 --imgsz 640 --batch auto --device 0 --name cigarette_v2

## E. Indir (best.pt -> PC)

In [ ]:
from google.colab import files
files.download('/content/runs/weapon_v3/weights/best.pt')